# Supervised Learning III — Logistic Regression

## References

Chapters 4.7.2, [ISLP] An Introduction to Statistical Learning - with Applications in Python. Free access to download the book: https://www.statlearning.com/

<a target="_blank" href="https://colab.research.google.com/github/cspun/MLF/blob/JNB/sup-III.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
np.random.seed(1)

# Display options
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

In Python:
- `statsmodels` for GLM-style logistic regression (coefficients, standard errors, p-values).
- `scikit-learn` for data splits, regularized logistic regression, cross-validation, calibration (Platt scaling), LDA, QDA, Naive Bayes.

## Logistic Regression (Smarket)
We fit a logistic regression to predict `Direction` using `Lag1`–`Lag5` and `Volume`.

In [3]:
# pip install statsmodels

In [4]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Load Smarket from R datasets via statsmodels
smarket = sm.datasets.get_rdataset("Smarket", package="ISLR").data.copy()
smarket["Direction"] = smarket["Direction"].astype("category")
smarket.head()

,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction
0,2001,0.3810,-0.1920,-2.6240,-1.0550,5.0100,1.1913,0.9590,Up
1,2001,0.9590,0.3810,-0.1920,-2.6240,-1.0550,1.2965,1.0320,Up
2,2001,1.0320,0.9590,0.3810,-0.1920,-2.6240,1.4112,-0.6230,Down
3,2001,-0.6230,1.0320,0.9590,0.3810,-0.1920,1.2760,0.6140,Up
4,2001,0.6140,-0.6230,1.0320,0.9590,0.3810,1.2057,0.2130,Up


Fit logistic regression with `statsmodels` (GLM binomial), analogous to `glm(..., family=binomial)` in R.

In [5]:
glm_fits = smf.glm(
    formula="Direction ~ Lag1 + Lag2 + Lag3 + Lag4 + Lag5 + Volume",
    data=smarket, family=sm.families.Binomial()
).fit()
print(glm_fits.summary())

                          Generalized Linear Model Regression Results                           
Dep. Variable:     ['Direction[Down]', 'Direction[Up]']   No. Observations:                 1250
Model:                                              GLM   Df Residuals:                     1243
Model Family:                                  Binomial   Df Model:                            6
Link Function:                                    Logit   Scale:                          1.0000
Method:                                            IRLS   Log-Likelihood:                -863.79
Date:                                  Sun, 14 Sep 2025   Deviance:                       1727.6
Time:                                          01:52:13   Pearson chi2:                 1.25e+03
No. Iterations:                                       4   Pseudo R-squ. (CS):           0.002868
Covariance Type:                              nonrobust                                         
                 coef    std e

Extract coefficients and p-values (similar to `coef()` and `summary(... )$coef` in R).

In [6]:
coefs = glm_fits.params
pvals = glm_fits.pvalues
display(coefs)
display(pvals)

Intercept    0.1260
Lag1         0.0731
Lag2         0.0423
Lag3        -0.0111
Lag4        -0.0094
Lag5        -0.0103
Volume      -0.1354
dtype: float64

Intercept   0.6007
Lag1        0.1452
Lag2        0.3984
Lag3        0.8243
Lag4        0.8514
Lag5        0.8350
Volume      0.3924
dtype: float64

Predicted probabilities on training data (`type="response"` in R).  
Check the coding of `Direction` (which class is treated as success *Up* vs *Down*): `statsmodels` uses the first (alphabetical) as 0; we will inspect explicitly by mapping labels.

In [7]:
glm_probs = glm_fits.predict(smarket)
glm_probs[:10]

0   0.4929
1   0.5185
2   0.5189
3   0.4848
4   0.4892
5   0.4930
6   0.5073
7   0.4908
8   0.4824
9   0.5112
dtype: float64

Convert probabilities to class labels at 0.5 threshold and compute confusion matrix & accuracy (training).

In [8]:
from sklearn.metrics import confusion_matrix, accuracy_score

pred_label = np.where(glm_probs > 0.5, "Up", "Down")
cm = confusion_matrix(smarket["Direction"], pred_label, labels=["Down", "Up"])
acc = accuracy_score(smarket["Direction"], pred_label)

print(pd.DataFrame(cm, index=["Actual Down","Actual Up"], columns=["Pred Down","Pred Up"]))
print("Training accuracy:", f"{acc:0.4f}")

             Pred Down  Pred Up
Actual Down        457      145
Actual Up          507      141
Training accuracy: 0.4784


### Train/Test Split: Train on 2001–2004, Test on 2005
(Train mask: `Year < 2005`)

In [9]:
train_mask = smarket["Year"] < 2005
test_mask = ~train_mask

smarket_2005 = smarket.loc[test_mask].copy()
direction_2005 = smarket_2005["Direction"].copy()

glm_fits_tt = smf.glm(
    formula="Direction ~ Lag1 + Lag2 + Lag3 + Lag4 + Lag5 + Volume",
    data=smarket.loc[train_mask], family=sm.families.Binomial()
).fit()

glm_probs_2005 = glm_fits_tt.predict(smarket_2005)
glm_pred_2005 = np.where(glm_probs_2005 > 0.5, "Up", "Down")

cm_2005 = confusion_matrix(direction_2005, glm_pred_2005, labels=["Down","Up"])
acc_2005 = accuracy_score(direction_2005, glm_pred_2005)

print(pd.DataFrame(cm_2005, index=["Actual Down","Actual Up"], columns=["Pred Down","Pred Up"]))
print("Test accuracy (2005):", f"{acc_2005:0.4f}", " | Test error:", f"{1-acc_2005:0.4f}")

             Pred Down  Pred Up
Actual Down         34       77
Actual Up           44       97
Test accuracy (2005): 0.5198  | Test error: 0.4802


Refit a reduced model using only `Lag1` and `Lag2` (as in the R notes), evaluate on 2005.

In [10]:
glm_fits_reduced = smf.glm(
    formula="Direction ~ Lag1 + Lag2",
    data=smarket.loc[train_mask], family=sm.families.Binomial()
).fit()

glm_probs_2005_r = glm_fits_reduced.predict(smarket_2005)
glm_pred_2005_r = np.where(glm_probs_2005_r > 0.5, "Up", "Down")

cm_2005_r = confusion_matrix(direction_2005, glm_pred_2005_r, labels=["Down","Up"])
acc_2005_r = accuracy_score(direction_2005, glm_pred_2005_r)

print(pd.DataFrame(cm_2005_r, index=["Actual Down","Actual Up"], columns=["Pred Down","Pred Up"]))
print("Reduced (Lag1+Lag2) — Test accuracy (2005):", f"{acc_2005_r:0.4f}")

             Pred Down  Pred Up
Actual Down         76       35
Actual Up          106       35
Reduced (Lag1+Lag2) — Test accuracy (2005): 0.4405


Predict probabilities for specific `Lag1`, `Lag2` pairs (mimicking R `predict(..., newdata=...)`).

In [11]:
new_df = pd.DataFrame({"Lag1":[1.2, 1.5], "Lag2":[1.1, -0.8]})
glm_fits_reduced.predict(new_df)

0   0.5209
1   0.5039
dtype: float64

## Regularized Logistic Regression (Ridge & Lasso)
We use `scikit-learn`'s `LogisticRegressionCV` with cross-validation to pick the regularization strength.  
**Standardization reminder:** Penalized models are sensitive to feature scale. We use a `Pipeline` with `StandardScaler` (fitted on training data only) → `LogisticRegressionCV`.

Predictors: `Lag1`–`Lag5`, `Volume`. Train: `Year < 2005`, Test: 2005.

In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import Pipeline

X = smarket[["Lag1","Lag2","Lag3","Lag4","Lag5","Volume"]].to_numpy()
y = smarket["Direction"].to_numpy()

X_train, X_test = X[train_mask.values], X[test_mask.values]
y_train, y_test = y[train_mask.values], y[test_mask.values]

# Ridge-penalized logistic (L2), CV selection of C (inverse of lambda)
ridge_clf = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("logit_cv", LogisticRegressionCV(
        Cs=20, cv=10, penalty="l2", solver="lbfgs", max_iter=1000,
        scoring="accuracy", refit=True, n_jobs=None
    ))
]).fit(X_train, y_train)

ridge_pred = ridge_clf.predict(X_test)
print("Ridge (L2) — Test accuracy:", accuracy_score(y_test, ridge_pred))

# Lasso-penalized logistic (L1) uses 'saga' solver
lasso_clf = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("logit_cv", LogisticRegressionCV(
        Cs=20, cv=10, penalty="l1", solver="saga", max_iter=2000,
        scoring="accuracy", refit=True, n_jobs=None
    ))
]).fit(X_train, y_train)

lasso_pred = lasso_clf.predict(X_test)
print("Lasso (L1) — Test accuracy:", accuracy_score(y_test, lasso_pred))

Ridge (L2) — Test accuracy: 0.5595238095238095
Lasso (L1) — Test accuracy: 0.5595238095238095


## Platt Scaling (Probability Calibration)
Platt scaling fits a sigmoid (logistic) mapping from raw classifier scores to calibrated probabilities.  
We demonstrate with `CalibratedClassifierCV(method='sigmoid')` on a ridge logistic base model.  
Calibration must use a **held-out** validation split (done internally by `CalibratedClassifierCV` via CV).

In [13]:
from sklearn.calibration import CalibratedClassifierCV

base_ridge = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("logit", LogisticRegressionCV(
        Cs=20, cv=10, penalty="l2", solver="lbfgs",
        max_iter=1000, scoring="neg_log_loss", refit=True
    ))
])

calibrated = CalibratedClassifierCV(estimator=base_ridge, method="sigmoid", cv=5)
calibrated.fit(X_train, y_train)
proba_uncal = base_ridge.fit(X_train, y_train).predict_proba(X_test)[:, 1]
proba_cal = calibrated.predict_proba(X_test)[:, 1]

pd.DataFrame({"uncalibrated": proba_uncal[:10], "calibrated": proba_cal[:10]})

,uncalibrated,calibrated
0,0.5085,0.4996
1,0.5082,0.4931
2,0.5084,0.4814
3,0.5081,0.4935
4,0.5077,0.5226
5,0.5078,0.5260
6,0.5079,0.5212
7,0.5081,0.5050
8,0.5079,0.5094
9,0.5081,0.5021


## Multinomial Logistic Regression (Iris)
Unregularized multinomial via `nnet::multinom` in R → here we use `statsmodels` MNLogit for summaries,  
then regularized multinomial via `LogisticRegressionCV`.

In [14]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import statsmodels.api as sm
import pandas as pd
import numpy as np
import warnings

# Silence only statsmodels' runtime sqrt warnings (optional)
warnings.filterwarnings("ignore", category=RuntimeWarning, module="statsmodels")

# Load data
iris = load_iris(as_frame=True)
X_iris = iris.data.copy()
y_num  = iris.target.copy()              # 0=setosa, 1=versicolor, 2=virginica
label_names = iris.target_names          # array(['setosa','versicolor','virginica'], dtype='<U10')

# Stratified split on numeric labels
Xtr, Xte, ytr, yte = train_test_split(
    X_iris, y_num, test_size=0.30, random_state=1, stratify=y_num
)

# Add intercept
X_sm = sm.add_constant(Xtr, has_constant="add")

# Try exact MLE with robust covariance to avoid NaN bse; fall back to gentle ridge if needed
try:
    mle_res = sm.MNLogit(ytr, X_sm).fit(method="newton", maxiter=400, disp=False)
    # Use heteroskedasticity-robust covariance (avoids invalid sqrt in SEs)
    mle_robust = mle_res.get_robustcov_results(cov_type="HC3")
    print(mle_robust.summary())
    mnlogit = mle_res  # use MLE for prediction
except Exception:
    # Gentle ridge (no L1) fallback for numerical stability; skip summary (SEs not defined the same way)
    mnlogit = sm.MNLogit(ytr, X_sm).fit_regularized(
        alpha=1e-8,      # very small ridge
        L1_wt=0.0,       # pure L2
        maxiter=4000,
        cnvrg_tol=1e-10,
        trim_mode="off",
        disp=False
    )
    print("Fitted MNLogit with tiny ridge regularization (alpha=1e-8, L2).")

# Predictions & confusion matrix
X_te_sm = sm.add_constant(Xte, has_constant="add")
proba = mnlogit.predict(X_te_sm)                    # shape: (n_test, n_classes)
pred_class = np.asarray(proba).argmax(axis=1)       # integer class indices 0..2
pred_name = label_names[pred_class]
true_name = label_names[yte]

print(pd.crosstab(pd.Series(pred_name, name="Pred"),
                  pd.Series(true_name, name="Actual")))
print("Accuracy:", accuracy_score(true_name, pred_name))

Fitted MNLogit with tiny ridge regularization (alpha=1e-8, L2).
Actual      setosa  versicolor  virginica
Pred                                     
setosa          15           0          0
versicolor       0          15          0
virginica        0           0         15
Accuracy: 1.0


C:\Miniconda3\envs\ds\Lib\site-packages\statsmodels\base\l1_solvers_common.py:71: ConvergenceWarning: QC check did not pass for 10 out of 10 parameters
Try increasing solver accuracy or number of iterations, decreasing alpha, or switch solvers
  warnings.warn(message, ConvergenceWarning)


### Regularized Multinomial Logistic Regression (Ridge & Lasso via scikit-learn)
Use `LogisticRegressionCV` with `multi_class='multinomial'` and `solver='lbfgs'` for Ridge, `solver='saga'` for Lasso.  
**Standardize** features in a `Pipeline`.

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler

# Ridge (L2) multinomial
ridge_multi = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("logit_cv", LogisticRegressionCV(
        Cs=20, cv=10, penalty="l2", solver="lbfgs",
        max_iter=2000, scoring="accuracy", refit=True
    ))
]).fit(Xtr, ytr)

ridge_multi_pred = ridge_multi.predict(Xte)
print("Ridge multinomial — Accuracy:", accuracy_score(yte, ridge_multi_pred))
print(pd.crosstab(pd.Series(ridge_multi_pred, name="Pred"), pd.Series(yte, name="Actual")))

# Lasso (L1) multinomial — requires 'saga'
lasso_multi = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("logit_cv", LogisticRegressionCV(
        Cs=20, cv=10, penalty="l1", solver="saga",
        max_iter=4000, scoring="accuracy", refit=True
    ))
]).fit(Xtr, ytr)

lasso_multi_pred = lasso_multi.predict(Xte)
print("Lasso multinomial — Accuracy:", accuracy_score(yte, lasso_multi_pred))
print(pd.crosstab(pd.Series(lasso_multi_pred, name="Pred"), pd.Series(yte, name="Actual")))

# Inspect sparsity (non-zero coefficients) for lasso
lr_lasso = lasso_multi.named_steps["logit_cv"]
coef_l1 = lr_lasso.coef_  # shape: (n_classes, n_features)
nonzeros_per_class = (coef_l1 != 0).sum(axis=1)
print("Non-zero features per class (lasso):", nonzeros_per_class)

Ridge multinomial — Accuracy: 1.0
Actual  0
Pred     
0       5
1       4
2       5
Lasso multinomial — Accuracy: 0.9777777777777777
Actual  0
Pred     
0       5
1       4
2       5
Non-zero features per class (lasso): [2 0 3]


## (Self-Study) Linear Discriminant Analysis (LDA)
Train on `Year < 2005` with predictors `Lag1`, `Lag2`. Evaluate on 2005.

In [16]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

X_lda = smarket[["Lag1","Lag2"]].to_numpy()
y_lda = smarket["Direction"].to_numpy()
Xtr_lda, Xte_lda = X_lda[train_mask.values], X_lda[test_mask.values]
ytr_lda, yte_lda = y_lda[train_mask.values], y_lda[test_mask.values]

lda = LinearDiscriminantAnalysis().fit(Xtr_lda, ytr_lda)
lda_pred = lda.predict(Xte_lda)
print(pd.crosstab(pd.Series(lda_pred, name="Pred"), pd.Series(yte_lda, name="Actual")))
print("LDA — Test accuracy:", accuracy_score(yte_lda, lda_pred))

Actual  Down   Up
Pred             
Down      35   35
Up        76  106
LDA — Test accuracy: 0.5595238095238095


## (Self-Study) Quadratic Discriminant Analysis (QDA)

In [17]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

qda = QuadraticDiscriminantAnalysis().fit(Xtr_lda, ytr_lda)
qda_pred = qda.predict(Xte_lda)
print(pd.crosstab(pd.Series(qda_pred, name="Pred"), pd.Series(yte_lda, name="Actual")))
print("QDA — Test accuracy:", accuracy_score(yte_lda, qda_pred))

Actual  Down   Up
Pred             
Down      30   20
Up        81  121
QDA — Test accuracy: 0.5992063492063492


## (Self-Study) Naive Bayes

In [18]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB().fit(Xtr_lda, ytr_lda)
nb_pred = nb.predict(Xte_lda)
print(pd.crosstab(pd.Series(nb_pred, name="Pred"), pd.Series(yte_lda, name="Actual")))
print("Naive Bayes — Test accuracy:", accuracy_score(yte_lda, nb_pred))

# Probability estimates (class posteriors)
nb_proba = nb.predict_proba(Xte_lda)[:5]
pd.DataFrame(nb_proba, columns=nb.classes_)

Actual  Down   Up
Pred             
Down      29   20
Up        82  121
Naive Bayes — Test accuracy: 0.5952380952380952


,Down,Up
0,0.4873,0.5127
1,0.4762,0.5238
2,0.4653,0.5347
3,0.4748,0.5252
4,0.4902,0.5098
